Anineko restartable Bulgarian subtitle worker. Enable Kaggle GPU and configure `ANINEKO_API_URL` and `ANINEKO_WORKER_TOKEN` in Kaggle Secrets.

In [ ]:
# Kaggle: enable GPU, add Kaggle Secrets ANINEKO_API_URL and ANINEKO_WORKER_TOKEN.
import os, sys, re, time, threading, subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.44,<5", "sentencepiece", "requests"])
import requests, torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from concurrent.futures import ThreadPoolExecutor
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
API = secrets.get_secret("ANINEKO_API_URL").rstrip("/")
TOKEN = secrets.get_secret("ANINEKO_WORKER_TOKEN")
MODEL = os.environ.get("SUBTITLE_MODEL", "facebook/nllb-200-distilled-600M")
WORKER = os.environ.get("KAGGLE_KERNEL_ID", "kaggle-worker")[:80]
STOP_AT = time.time() + 10 * 60 * 60 - 600
HEADERS = {"Authorization": "Bearer " + TOKEN, "Content-Type": "application/json", "User-Agent": f"AninekoKaggleWorker/1.0 ({WORKER})"}
HEARTBEAT_STOP = threading.Event()
LEASE_LOST = threading.Event()
LEASE_LAST_OK = [time.monotonic()]
if not API.startswith("https://") or len(TOKEN) < 32: raise RuntimeError("Configure valid HTTPS API URL and worker token in Kaggle Secrets")
if not torch.cuda.is_available(): raise RuntimeError("Kaggle GPU runtime is required")
GPU_COUNT = torch.cuda.device_count()
print({"model": MODEL, "gpus": GPU_COUNT, "devices": [torch.cuda.get_device_name(i) for i in range(GPU_COUNT)]})

def heartbeat_job(job_id):
    while not HEARTBEAT_STOP.wait(30):
        try:
            response=requests.post(API+"/api/worker/heartbeat",headers=HEADERS,json={"jobId":job_id,"workerId":WORKER},timeout=15)
            if response.status_code in (401,403):
                LEASE_LOST.set(); return
            response.raise_for_status()
            if not response.json().get("ok"):
                LEASE_LOST.set(); return
            LEASE_LAST_OK[0]=time.monotonic()
        except requests.RequestException:
            if time.monotonic()-LEASE_LAST_OK[0]>=90:
                LEASE_LOST.set(); return

def parse_srt(body):
    body = body.lstrip("\ufeff").replace("\r\n", "\n").replace("\r", "\n")
    cues = []
    for block in re.split(r"\n\s*\n", body.strip()):
        lines = block.split("\n")
        ti = next((i for i,x in enumerate(lines) if "-->" in x), None)
        if ti is None or ti == len(lines)-1: continue
        m = re.match(r"\s*(\d{1,2}:\d{2}:\d{2}[,.]\d{1,3})\s*-->\s*(\d{1,2}:\d{2}:\d{2}[,.]\d{1,3})(.*)", lines[ti])
        if not m: raise ValueError("Malformed SRT timing")
        cues.append({"id": lines[ti-1].strip() if ti else str(len(cues)+1), "start":m.group(1), "end":m.group(2), "settings":m.group(3), "text":"\n".join(lines[ti+1:]).strip()})
    if not cues: raise ValueError("No subtitle cues")
    return cues

def serialize(cues, translated):
    return "\n\n".join(f"{c['id']}\n{c['start']} --> {c['end']}{c['settings']}\n{t}" for c,t in zip(cues,translated)) + "\n"

LANG = {"en":"eng_Latn", "ja":"jpn_Jpan"}
# Independent model instances on each GPU; batches are assigned round-robin, outputs restored to cue order.
models=[]
for device in range(GPU_COUNT):
    tok=AutoTokenizer.from_pretrained(MODEL, src_lang="eng_Latn")
    model=AutoModelForSeq2SeqLM.from_pretrained(MODEL, torch_dtype=torch.float16).to(f"cuda:{device}").eval()
    models.append((tok,model,f"cuda:{device}"))

def translate_chunk(chunk, source, gpu):
    tok, model, device=models[gpu]
    tok.src_lang=LANG[source]
    texts=[re.sub(r"<[^>]+>|\{[^}]*\}", "", c["text"]) for c in chunk]
    inputs=tok(texts, return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
    try:
        with torch.inference_mode():
            out=model.generate(**inputs, forced_bos_token_id=tok.convert_tokens_to_ids("bul_Cyrl"), max_new_tokens=160, num_beams=2)
        return [tok.decode(x, skip_special_tokens=True).strip() for x in out]
    except RuntimeError as error:
        if "out of memory" not in str(error).lower() or len(chunk) <= 1: raise
        del inputs
        torch.cuda.empty_cache()
        midpoint=len(chunk)//2
        return translate_chunk(chunk[:midpoint],source,gpu)+translate_chunk(chunk[midpoint:],source,gpu)

def translate_job(job):
    cues=parse_srt(job["sourceText"])
    if len(cues)!=len(job["cues"]): raise ValueError("backend cue count differs from source")
    source=job["sourceLanguage"]
    if source not in LANG: raise ValueError("unsupported source language: "+str(source))
    translated=[None]*len(cues)
    partial=parse_srt(job["partialText"]) if job.get("partialText", "").strip() else []
    if len(partial)>len(cues): raise ValueError("checkpoint contains more cues than source")
    for i,cue in enumerate(partial):
        if cue["start"]!=cues[i]["start"] or cue["end"]!=cues[i]["end"]: raise ValueError("checkpoint timestamps differ from source")
        translated[i]=cue["text"]
    batch_size=16
    remaining=list(range(len(partial),len(cues)))
    chunks=[remaining[i:i+batch_size] for i in range(0,len(remaining),batch_size)]
    for start in range(0,len(chunks),GPU_COUNT):
        futures=[]
        with ThreadPoolExecutor(max_workers=min(GPU_COUNT,len(chunks)-start)) as pool:
            for j in range(start,min(start+GPU_COUNT,len(chunks))):
                indices=chunks[j]
                futures.append((indices,pool.submit(translate_chunk,[cues[i] for i in indices],source,j%GPU_COUNT)))
            for indices,future in futures:
                values=future.result()
                if len(values)!=len(indices): raise ValueError("model output cue parity mismatch")
                for i,value in zip(indices,values): translated[i]=value
        completed=next((i for i in range(len(translated)-1,-1,-1) if translated[i] is not None),-1)+1
        partial_result=serialize(cues[:completed],translated[:completed])
        if LEASE_LOST.is_set(): raise RuntimeError("worker lease lost before checkpoint")
        requests.post(API+"/api/worker/checkpoint",headers=HEADERS,json={"jobId":job["id"],"workerId":WORKER,"translatedText":partial_result},timeout=60).raise_for_status()
        LEASE_LAST_OK[0]=time.monotonic()
    if any(not x for x in translated): raise ValueError("empty translated cue")
    cyr=sum(1 for x in translated for ch in x if "\u0400"<=ch<="\u04ff")
    letters=sum(1 for x in translated for ch in x if ch.isalpha())
    if not letters or cyr/letters < .5: raise ValueError("Bulgarian Cyrillic ratio below 0.5")
    result=serialize(cues,translated)
    payload={"jobId":job["id"],"workerId":WORKER,"translatedText":result,"model":MODEL,"translationVersion":"kaggle-nllb-v1"}
    if LEASE_LOST.is_set(): raise RuntimeError("worker lease lost before completion")
    requests.post(API+"/api/worker/complete",headers=HEADERS,json=payload,timeout=120).raise_for_status()
    return len(cues)

completed=0; cue_total=0; failures=0; started=time.time()
while time.time()<STOP_AT:
    job=None
    try:
        response=requests.post(API+"/api/worker/claim",headers=HEADERS,json={"workerId":WORKER},timeout=150)
        response.raise_for_status(); job=response.json().get("job")
        if not job: time.sleep(20); continue
        HEARTBEAT_STOP.clear(); LEASE_LOST.clear(); LEASE_LAST_OK[0]=time.monotonic()
        heartbeat=threading.Thread(target=heartbeat_job,args=(job["id"],),daemon=True); heartbeat.start()
        try:
            n=translate_job(job); completed+=1; cue_total+=n
        finally:
            HEARTBEAT_STOP.set(); heartbeat.join(timeout=16)
        print({"event":"completed","animeId":job["animeId"],"episode":job["episodeId"],"cues":n,"episodesPerHour":completed*3600/max(1,time.time()-started)})
    except Exception as e:
        failures+=1
        try:
            if "job" in locals() and job:
                requests.post(API+"/api/worker/fail",headers=HEADERS,json={"jobId":job["id"],"workerId":WORKER,"reason":str(e)[:900]},timeout=20)
        except Exception: pass
        print({"event":"retry","failures":failures,"error":str(e)[:300]})
        time.sleep(min(120,5*2**min(failures,4)))
print({"event":"session_stopped","episodes":completed,"cues":cue_total,"failures":failures,"seconds":time.time()-started,"gpus":GPU_COUNT})